## Neuronale Netze

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from keras.models import Sequential
from keras.optimizers import Adam


# Setze Zufallssamen
initial_seed = 42
np.random.seed(initial_seed)
tf.random.set_seed(initial_seed)

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [4]:
# Globales DataFrame zur Speicherung der Ergebnisse
log_results_df = pd.DataFrame(columns=["Durchlauf", "Modell", "Accuracy", "F1 Score"])

def log_model_performance(durchlauf_name: str, model_name: str, y_pred, y_test, seed: int):
    """
    Berechnet Accuracy und F1-Score anhand von y_test und y_pred und
    fügt die Ergebnisse dem globalen DataFrame results_df hinzu.

    Parameter:
      - durchlauf_name: Name des Durchlaufs (String)
      - model_name: Name des verwendeten Modells (String)
      - y_pred: Vorhersagen des Modells
      - y_test: Wahre Zielwerte

    Gibt zurück:
      - Das aktualisierte DataFrame mit den Spalten "Durchlauf", "Modell", "Accuracy" und "F1 Score"
    """
    global log_results_df
    # Berechnung der Metriken
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Neuen Eintrag als DataFrame erstellen
    new_row = pd.DataFrame({
        "Durchlauf": [durchlauf_name],
        "Modell": [model_name],
        "Seed": [seed],
        "Accuracy": [acc],
        "F1 Score": [f1]
    })

    # Ergebnisse zum globalen DataFrame hinzufügen
    log_results_df = pd.concat([log_results_df, new_row], ignore_index=True)
    return log_results_df

In [5]:
# Initialisiere das globale DataFrame
log_results_df = pd.DataFrame(columns=["Durchlauf", "Modell", "Seed", "Layers", "Neurons", "Dropout Rate", "Batch Normalization", "Accuracy", "F1 Score"])

def log_model_performance2(durchlauf_name: str, model_name: str, y_pred, y_test,
                          seed: int = None, layers: int = None, neurons: int = None,
                          dropout_rate: float = None, batch_norm: bool = None):
    """
    Berechnet Accuracy und F1-Score anhand von y_test und y_pred und
    fügt die Ergebnisse dem globalen DataFrame log_results_df hinzu.

    Parameter:
      - durchlauf_name: Name des Durchlaufs (String)
      - model_name: Name des verwendeten Modells (String)
      - y_pred: Vorhersagen des Modells
      - y_test: Wahre Zielwerte
      - seed: Verwendeter Seed (optional)
      - layers: Anzahl der Schichten (optional)
      - neurons: Anzahl der Neuronen pro Schicht (optional)
      - dropout_rate: Dropout-Rate (optional)
      - batch_norm: Verwendung von Batch Normalization (optional)
      - accuracy: Berechnete Accuracy (optional)
      - f1_score: Berechneter F1-Score (optional)

    Gibt zurück:
      - Das aktualisierte DataFrame mit den Spalten "Durchlauf", "Modell", "Seed", "Layers", "Neurons", "Dropout Rate", "Batch Normalization", "Accuracy" und "F1 Score"
    """
    global log_results_df

    # Berechnung der Metriken, falls nicht bereits übergeben

    accuracy = accuracy_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    # Neuen Eintrag als DataFrame erstellen
    new_row = pd.DataFrame({
        "Durchlauf": [durchlauf_name],
        "Modell": [model_name],
        "Seed": [seed if seed is not None else "NA"],
        "Layers": [layers if layers is not None else "NA"],
        "Neurons": [neurons if neurons is not None else "NA"],
        "Dropout Rate": [dropout_rate if dropout_rate is not None else "NA"],
        "Batch Normalization": [batch_norm if batch_norm is not None else "NA"],
        "Accuracy": [accuracy],
        "F1 Score": [f1]
    })

    # Ergebnisse zum globalen DataFrame hinzufügen
    log_results_df = pd.concat([log_results_df, new_row], ignore_index=True)
    return log_results_df

In [6]:
df_training_all_features = pd.read_parquet(
    "df_training_all_features.parquet"
)

In [ ]:
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Input(shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=0)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Baseline", "Neuronales Netz 2-Layers", y_pred_class, y_test)
df_results

Test accuracy: 0.5435897707939148
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


<ipython-input-52-e05d3ced303f>:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  log_results_df = pd.concat([log_results_df, new_row], ignore_index=True)


,Durchlauf,Modell,Accuracy,F1 Score
0,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.54359,0.636735


In [7]:
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline-Modellierung
def create_model(deep_layers, neurons, seed=None, dropout_rate=0.0, batch_norm=False):
    if seed is None:
        seed = np.random.randint(1, 100)  # Generiere einen zufälligen Seed
        print(f"No seed provided. Using random seed: {seed}")

    # Setze die Zufallsgeneratoren
    np.random.seed(seed)
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    for _ in range(deep_layers):
        model.add(Dense(neurons, activation='relu',kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))
        if batch_norm:
            model.add(BatchNormalization())
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    return model, seed

In [8]:
# Beispielaufruf der create_model-Funktion
deep_layers = 1  # Definiere die Anzahl der Schichten
neurons = 32     # Definiere die Anzahl der Neuronen
model, seed = create_model(deep_layers=deep_layers, neurons=neurons)


model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=0)


loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")


y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)

# Logge die Ergebnisse
df_results = log_model_performance2(f"Baseline", f"Neuronales Netz", y_pred_class, y_test, seed, deep_layers, neurons)
df_results


No seed provided. Using random seed: 52
Test accuracy: 0.5435897707939148
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


<ipython-input-5-2064eb4499b3>:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  log_results_df = pd.concat([log_results_df, new_row], ignore_index=True)


,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,Baseline,Neuronales Netz,52,1,32,NA,NA,0.54359,0.630705


In [9]:

# Beispielaufruf der create_model-Funktion
deep_layers = 2  # Definiere die Anzahl der Schichten
neurons = 32     # Definiere die Anzahl der Neuronen
model, seed = create_model(deep_layers=deep_layers, neurons=neurons, seed=initial_seed)


model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=0)


loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")


y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)

# Logge die Ergebnisse
df_results = log_model_performance2(f"Baseline", f"Neuronales Netz", y_pred_class, y_test, seed, deep_layers, neurons)
df_results


Test accuracy: 0.6051282286643982
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,Baseline,Neuronales Netz,52,1,32,NA,NA,0.543590,0.630705
1,Baseline,Neuronales Netz,42,2,32,NA,NA,0.605128,0.222222


In [10]:
# Beispielaufruf der create_model-Funktion
deep_layers = 3  # Definiere die Anzahl der Schichten
neurons = 64     # Definiere die Anzahl der Neuronen
model, seed = create_model(deep_layers=deep_layers, neurons=neurons, seed=initial_seed)


model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)


loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")


y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)

# Logge die Ergebnisse
df_results = log_model_performance2(f"100 Epo, 50 BatchS", f"Neuronales Netz", y_pred_class, y_test, seed, deep_layers, neurons)
df_results


Test accuracy: 0.6512820720672607
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,Baseline,Neuronales Netz,52,1,32,NA,NA,0.543590,0.630705
1,Baseline,Neuronales Netz,42,2,32,NA,NA,0.605128,0.222222
2,"100 Epo, 50 BatchS",Neuronales Netz,42,3,64,NA,NA,0.651282,0.433333


In [13]:
df = df_training_all_features#.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]



# One-Hot-Encoding für kategoriale Features
categorical_features = ["Sex", "Ctry", "Town", "Goal of Training", "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

deep_layers = 4  # Definiere die Anzahl der Schichten
neurons = 64     # Definiere die Anzahl der Neuronen
model, seed = create_model(deep_layers=deep_layers, neurons=neurons, seed)


model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)


loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")


y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)

# Logge die Ergebnisse
df_results = log_model_performance2(f"OneHot-Encoding", f"Neuronales Netz", y_pred_class, y_test, seed, deep_layers, neurons)
df_results

No seed provided. Using random seed: 86
Test accuracy: 0.7897436022758484
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,Baseline,Neuronales Netz,52,1,32,NA,NA,0.543590,0.630705
1,Baseline,Neuronales Netz,42,2,32,NA,NA,0.605128,0.222222
2,"100 Epo, 50 BatchS",Neuronales Netz,42,3,64,NA,NA,0.651282,0.433333
3,OneHot-Encoding,Neuronales Netz,52,3,64,NA,NA,0.558974,0.641667
4,OneHot-Encoding,Neuronales Netz,29,3,64,NA,NA,0.733333,0.711111
5,OneHot-Encoding,Neuronales Netz,86,4,64,NA,NA,0.789744,0.721088


In [14]:
model.save('model_OHEnc_86.keras')

In [16]:
#seeds = [42, 21, 100, 123, 500, 789, 999]
seeds = np.random.randint(1, 1000, size=20).tolist()

# Speichere die Ergebnisse in einem DataFrame
results_seeds = []

for seed in seeds:
    # Setze Zufallssamen
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Erstelle das Modell
    model, seed = create_model(deep_layers=3, neurons=64, seed=seed)

    # Kompilieren des Modells
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Trainieren des Modells
    #stop_callback = StopTrainingCallback()
    model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)

    # Evaluiere das Modell
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Seed {seed}: Test accuracy = {accuracy}")

    # Speichere die Ergebnisse
    results_seeds.append({"Seed": seed, "Test Accuracy": accuracy})

Seed 939: Test accuracy = 0.7333333492279053
Seed 754: Test accuracy = 0.656410276889801
Seed 858: Test accuracy = 0.5230769515037537
Seed 313: Test accuracy = 0.6717948913574219
Seed 502: Test accuracy = 0.6871795058250427
Seed 381: Test accuracy = 0.6871795058250427
Seed 207: Test accuracy = 0.6000000238418579
Seed 110: Test accuracy = 0.7179487347602844
Seed 577: Test accuracy = 0.7230769395828247
Seed 239: Test accuracy = 0.5487179756164551
Seed 994: Test accuracy = 0.7076923251152039
Seed 715: Test accuracy = 0.692307710647583
Seed 763: Test accuracy = 0.5538461804389954
Seed 709: Test accuracy = 0.5743589997291565
Seed 674: Test accuracy = 0.7230769395828247
Seed 890: Test accuracy = 0.6871795058250427
Seed 470: Test accuracy = 0.5692307949066162
Seed 78: Test accuracy = 0.7076923251152039
Seed 58: Test accuracy = 0.5589743852615356
Seed 125: Test accuracy = 0.6512820720672607


In [18]:
# Experimentiere mit verschiedenen Parametern
param_opt = []
seed = 86

deep_layers = [3, 4]
neurons = [64, 128]
dropout_rates = [0.0, 0.2,]
batch_norm_options = [True, False]


np.random.seed(seed)
tf.random.set_seed(seed)
for layer in deep_layers:
    for neuron in neurons:
        for dropout_rate in dropout_rates:
            for batch_norm in batch_norm_options:
                model, seed = create_model(seed=seed, deep_layers=layer, neurons=neuron, dropout_rate=dropout_rate, batch_norm=batch_norm)
                model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
                model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)
                loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
                param_opt = log_model_performance2(f"Ganzheitliche Optimierung", f"Neuronales Netz", y_pred_class, y_test, seed, layer, neuron, dropout_rate, batch_norm)


param_opt

,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,Baseline,Neuronales Netz,52,1,32,NA,NA,0.543590,0.630705
1,Baseline,Neuronales Netz,42,2,32,NA,NA,0.605128,0.222222
2,"100 Epo, 50 BatchS",Neuronales Netz,42,3,64,NA,NA,0.651282,0.433333
3,OneHot-Encoding,Neuronales Netz,52,3,64,NA,NA,0.558974,0.641667
4,OneHot-Encoding,Neuronales Netz,29,3,64,NA,NA,0.733333,0.711111
5,OneHot-Encoding,Neuronales Netz,86,4,64,NA,NA,0.789744,0.721088
6,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.0,True,0.789744,0.721088
7,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.0,False,0.789744,0.721088
8,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.2,True,0.789744,0.721088
9,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.2,False,0.789744,0.721088


In [22]:
deep_layers = 3  # Definiere die Anzahl der Schichten
neurons = 64     # Definiere die Anzahl der Neuronen
model, seed = create_model(deep_layers=deep_layers, neurons=neurons, seed=seed)

results_fit_param=[]

epochs_options = [50, 100, 200]
batch_size_options = [10, 32, 64, 128]

for epochs in epochs_options:
        for batch_size in batch_size_options:
            model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
            model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)
            loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

            loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
            y_pred = model.predict(X_test)
            y_pred_class = (y_pred > 0.5).astype(int)

            results_fit_param.append({
                "Seed": seed,
                "Epochs": epochs,
                "Batch Size": batch_size,
                "Test Accuracy": accuracy,
                "F1 Score": f1_score(y_test, y_pred_class)
            })
results_fit_param = pd.DataFrame(results_fit_param)
results_fit_param

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


,Seed,Epochs,Batch Size,Test Accuracy,F1 Score
0,86,50,10,0.646154,0.666667
1,86,50,32,0.748718,0.631579
2,86,50,64,0.482051,0.613027
3,86,50,128,0.733333,0.617647
4,86,100,10,0.769231,0.701987
5,86,100,32,0.758974,0.728324
6,86,100,64,0.743590,0.657534
7,86,100,128,0.743590,0.647887
8,86,200,10,0.620513,0.339286
9,86,200,32,0.758974,0.680272


In [20]:
seed

86

In [24]:
deep_layers = 4  # Definiere die Anzahl der Schichten
neurons = 64     # Definiere die Anzahl der Neuronen
model, seed = create_model(deep_layers=deep_layers, neurons=neurons, seed=seed)


model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)


loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")


y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)

# Logge die Ergebnisse
df_results = log_model_performance2(f"Seed Optimierung", f"Neuronales Netz", y_pred_class, y_test, seed, deep_layers, neurons)
df_results

Test accuracy: 0.764102578163147
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,Baseline,Neuronales Netz,52,1,32,NA,NA,0.543590,0.630705
1,Baseline,Neuronales Netz,42,2,32,NA,NA,0.605128,0.222222
2,"100 Epo, 50 BatchS",Neuronales Netz,42,3,64,NA,NA,0.651282,0.433333
3,OneHot-Encoding,Neuronales Netz,52,3,64,NA,NA,0.558974,0.641667
4,OneHot-Encoding,Neuronales Netz,29,3,64,NA,NA,0.733333,0.711111
5,OneHot-Encoding,Neuronales Netz,86,4,64,NA,NA,0.789744,0.721088
6,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.0,True,0.789744,0.721088
7,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.0,False,0.789744,0.721088
8,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.2,True,0.789744,0.721088
9,Ganzheitliche Optimierung,Neuronales Netz,86,3,64,0.2,False,0.789744,0.721088


# Test

In [ ]:
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Input(shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Baseline 50-Epo, 10-BatchS", "Neuronales Netz 3-Layers", y_pred_class, y_test)
df_results

Epoch 1/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5163 - loss: 43.9666 - val_accuracy: 0.5449 - val_loss: 4.8993
Epoch 2/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5523 - loss: 5.6177 - val_accuracy: 0.6346 - val_loss: 2.5015
Epoch 3/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6356 - loss: 2.5760 - val_accuracy: 0.6410 - val_loss: 1.4502
Epoch 4/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5869 - loss: 3.5971 - val_accuracy: 0.5513 - val_loss: 4.6849
Epoch 5/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5640 - loss: 4.2084 - val_accuracy: 0.5449 - val_loss: 3.7175
Epoch 6/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6310 - loss: 2.6727 - val_accuracy: 0.5962 - val_loss: 3.8977
Epoch 7/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5848 - loss: 3.9533 - val_accuracy: 0.6538 - val_loss: 2.4723
Epoch 8/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6084 - loss: 3.7738 - val_accuracy: 0.6282 - val_loss

Test accuracy: 0.5692307949066162
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


,Durchlauf,Modell,Accuracy,F1 Score
0,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.507692,0.625000
1,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.446154,0.600000
2,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.697436,0.628931
3,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.569231,0.142857


In [ ]:
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Input(shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Baseline 100-Epo, 50-BatchS", "Neuronales Netz 3-Layers", y_pred_class, y_test)
df_results

In [ ]:
# Daten aufbereiten
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Liste der Seeds
#seeds = [42, 21, 100, 123, 500, 789, 999]
seeds = np.random.randint(1, 10000, size=20).tolist()

# Speichere die Ergebnisse in einem DataFrame
results_seeds = []

for seed in seeds:
    # Setze Zufallssamen
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Erstelle das Modell
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))  # Verwende Input Layer
    model.add(Dense(64, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))
    model.add(Dense(32, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))
    model.add(Dense(1, activation='sigmoid', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))

    # Kompilieren des Modells
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Trainieren des Modells
    model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)

    # Evaluiere das Modell
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Seed {seed}: Test accuracy = {accuracy}")

    # Speichere die Ergebnisse
    results_seeds.append({"Seed": seed, "Test Accuracy": accuracy})

Seed 9137: Test accuracy = 0.728205144405365
Seed 6914: Test accuracy = 0.5948718190193176
Seed 5326: Test accuracy = 0.7128205299377441
Seed 9830: Test accuracy = 0.7128205299377441
Seed 5586: Test accuracy = 0.5230769515037537
Seed 9700: Test accuracy = 0.6051282286643982
Seed 7292: Test accuracy = 0.6820513010025024
Seed 7528: Test accuracy = 0.7179487347602844
Seed 5753: Test accuracy = 0.7230769395828247
Seed 1309: Test accuracy = 0.728205144405365
Seed 1253: Test accuracy = 0.5641025900840759
Seed 1204: Test accuracy = 0.7384615540504456
Seed 1443: Test accuracy = 0.6102564334869385
Seed 7182: Test accuracy = 0.6410256624221802
Seed 7655: Test accuracy = 0.6717948913574219
Seed 1444: Test accuracy = 0.7230769395828247
Seed 8234: Test accuracy = 0.7179487347602844
Seed 9333: Test accuracy = 0.7076923251152039
Seed 2692: Test accuracy = 0.5641025900840759
Seed 394: Test accuracy = 0.728205144405365


In [ ]:
results_seeds = pd.DataFrame(results_seeds)
best_seed = int(results_seeds.loc[results_seeds['Test Accuracy'].idxmax()]['Seed'])
print(f"Best Seed in this random run and env: \n{best_seed}")

Best Seed in this random run and env: 
1204


In [ ]:
# Daten aufbereiten
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Liste der Seeds
#seeds = [42, 21, 100, 123, 500, 789, 999]
seeds = np.random.randint(1, 10000, size=20).tolist()

# Speichere die Ergebnisse in einem DataFrame
results_seeds = []

for seed in seeds:
    # Setze Zufallssamen
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Erstelle das Modell
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))  # Verwende Input Layer
    model.add(Dense(64, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))
    model.add(Dense(32, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))
    model.add(Dense(1, activation='sigmoid', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed)))

    # Kompilieren des Modells
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Trainieren des Modells
    model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=0)

    # Evaluiere das Modell
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Seed {seed}: Test accuracy = {accuracy}")

    # Speichere die Ergebnisse
    results_seeds.append({"Seed": seed, "Test Accuracy": accuracy})

Seed 2955: Test accuracy = 0.7435897588729858
Seed 1186: Test accuracy = 0.6000000238418579
Seed 6790: Test accuracy = 0.5538461804389954
Seed 4343: Test accuracy = 0.6461538672447205
Seed 72: Test accuracy = 0.7435897588729858
Seed 3923: Test accuracy = 0.446153849363327
Seed 9845: Test accuracy = 0.5846154093742371
Seed 2106: Test accuracy = 0.692307710647583
Seed 2015: Test accuracy = 0.5435897707939148
Seed 2946: Test accuracy = 0.728205144405365
Seed 3667: Test accuracy = 0.7230769395828247
Seed 4640: Test accuracy = 0.6666666865348816
Seed 8351: Test accuracy = 0.6051282286643982
Seed 7077: Test accuracy = 0.5487179756164551
Seed 6297: Test accuracy = 0.5025641322135925
Seed 7328: Test accuracy = 0.7179487347602844
Seed 8007: Test accuracy = 0.7384615540504456
Seed 4217: Test accuracy = 0.5589743852615356
Seed 1991: Test accuracy = 0.692307710647583
Seed 7137: Test accuracy = 0.5743589997291565


In [ ]:
np.random.seed(best_seed)
tf.random.set_seed(best_seed)

categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Input(shape=(X_train.shape[1],)))  # Verwende Input Layer
model.add(Dense(64, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))
model.add(Dense(32, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))
model.add(Dense(1, activation='sigmoid', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Best Seed Opt. 100-Epo, 50-BatchS", "Neuronales Netz 3-Layers", y_pred_class, y_test, best_seed)
df_results

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5150 - loss: 36.8849 - val_accuracy: 0.5513 - val_loss: 9.4941
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5767 - loss: 8.1396 - val_accuracy: 0.5449 - val_loss: 10.0246
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5906 - loss: 5.5359 - val_accuracy: 0.6026 - val_loss: 3.1833
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6277 - loss: 3.5264 - val_accuracy: 0.6282 - val_loss: 1.8782
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6495 - loss: 2.0760 - val_accuracy: 0.6474 - val_loss: 1.8594
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6420 - loss: 1.4259 - val_accuracy: 0.6538 - val_loss: 1.0685
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6623 - loss: 1.2526 - val_accuracy: 0.6218 - val_loss: 1.5393
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6764 - loss: 1.2016 - val_accuracy: 0.5385 

,Durchlauf,Modell,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Accuracy,F1 Score
0,"Baseline, 2 Deep-Layers, 32 Neuronen, 50 Epo, 10 BatchS",Neuronales Netz,94,NA,NA,NA,NA,0.656410,0.446281
1,"Baseline, 2 Deep-Layers, 32 Neuronen, 50 Epo, 10 BatchS",Neuronales Netz,60,NA,NA,NA,NA,0.661538,0.679612
2,"Baseline, 50 Epo, 10 BatchS",Neuronales Netz,78,2,32,NA,NA,0.671795,0.695238
3,"Baseline, 50 Epo, 10 BatchS",Neuronales Netz,70,2,32,NA,NA,0.558974,0.638655
4,Optimization,Neuronales Netz,32,"[2, 3]","[32, 64]","[0.2, 0.3]","[True, False]",0.558974,0.638655
5,Optimization,Neuronales Netz,32,"[2, 3]","[32, 64]","[0.2, 0.3]","[True, False]",0.558974,0.638655
6,"Baseline, 100 Epo, 50 BatchS",Neuronales Netz,88,3,64,NA,NA,0.569231,0.045455
7,"Baseline, 100 Epo, 50 BatchS",Neuronales Netz,89,3,64,NA,NA,0.548718,0.633333
8,"Baseline, 100 Epo, 50 BatchS",Neuronales Netz,20,3,64,NA,NA,0.512821,0.615385
9,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,1204,NaN,NaN,NaN,NaN,0.738462,0.675159


In [ ]:
np.random.seed(best_seed)
tf.random.set_seed(best_seed)

categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Input(shape=(X_train.shape[1],)))  # Verwende Input Layer
model.add(Dense(64, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))
model.add(Dense(64, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))
model.add(Dense(32, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))
model.add(Dense(1, activation='sigmoid', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=best_seed)))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Best Seed Opt. 100-Epo, 50-BatchS", "Neuronales Netz 4-Layers", y_pred_class, y_test)
df_results

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.4385 - loss: 21.0318 - val_accuracy: 0.5449 - val_loss: 6.5444
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5650 - loss: 4.4714 - val_accuracy: 0.5449 - val_loss: 5.9100
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6434 - loss: 2.9248 - val_accuracy: 0.5449 - val_loss: 2.4647
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5911 - loss: 2.2566 - val_accuracy: 0.5705 - val_loss: 3.1689
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5574 - loss: 3.0917 - val_accuracy: 0.5064 - val_loss: 3.6188
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5762 - loss: 2.7867 - val_accuracy: 0.6154 - val_loss: 2.2194
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5985 - loss: 2.2068 - val_accuracy: 0.6731 - val_loss: 1.0261
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6711 - loss: 1.3281 - val_accuracy: 0.5064 

,Durchlauf,Modell,Accuracy,F1 Score
0,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.507692,0.625000
1,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.446154,0.600000
2,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.697436,0.628931
3,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.569231,0.142857
4,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.671795,0.457627
5,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
6,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.584615,0.198020
7,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.743590,0.691358
8,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
9,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 4-Layers,0.569231,0.066667


In [ ]:
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Liste der Seeds
seeds = [42, 21, 100, 123, 500, 789, 999]

# Speichere die Ergebnisse
results = []

for seed in seeds:
    # Setze Zufallssamen
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Erstelle das Modell
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    # Kompilieren des Modells
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Trainieren des Modells
    model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)

    # Evaluiere das Modell
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"Seed {seed}: Test accuracy = {accuracy}")

    # Speichere die Ergebnisse
    results.append((seed, accuracy))

# Ergebnisse ausgeben
for seed, accuracy in results:
    print(f"Seed {seed}: Test accuracy = {accuracy}")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Seed 42: Test accuracy = 0.7230769395828247


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Seed 21: Test accuracy = 0.5641025900840759


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

Seed 500: Test accuracy = 0.5897436141967773


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Seed 789: Test accuracy = 0.6871795058250427


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Seed 999: Test accuracy = 0.7025641202926636
Seed 42: Test accuracy = 0.7230769395828247
Seed 21: Test accuracy = 0.5641025900840759
Seed 100: Test accuracy = 0.6974359154701233
Seed 123: Test accuracy = 0.7128205299377441
Seed 500: Test accuracy = 0.5897436141967773
Seed 789: Test accuracy = 0.6871795058250427
Seed 999: Test accuracy = 0.7025641202926636


In [ ]:
# Lade dein Daten
# Liste der kategorialen Spalten
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features#.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]



# One-Hot-Encoding für kategoriale Features
categorical_features = ["Sex", "Ctry", "Town", "Goal of Training", "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# Feature Scaling
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Erstelle das neuronale Netz
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # Sigmoid-Funktion für binäre Klassifikation

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=50, batch_size=10, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("One-Hot Cat", "Neuronales Netz 3-Schichten 50 Epochen batchsize 10", y_pred_class, y_test)
df_results

Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5469 - loss: 18.4748 - val_accuracy: 0.5449 - val_loss: 9.9363
Epoch 2/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5537 - loss: 6.1453 - val_accuracy: 0.5449 - val_loss: 7.3740
Epoch 3/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6390 - loss: 3.1860 - val_accuracy: 0.6346 - val_loss: 1.2801
Epoch 4/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6464 - loss: 1.2297 - val_accuracy: 0.6090 - val_loss: 0.8413
Epoch 5/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6654 - loss: 1.2778 - val_accuracy: 0.5449 - val_loss: 2.8055
Epoch 6/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6165 - loss: 1.6911 - val_accuracy: 0.7115 - val_loss: 0.7312
Epoch 7/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6548 - loss: 1.1313 - val_accuracy: 0.5449 - val_loss: 2.0648
Epoch 8/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6367 - loss: 1.2618 - val_accuracy: 0.5449 - val_loss: 1.3642
Ep

,Durchlauf,Modell,Accuracy,F1 Score
0,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.507692,0.625000
1,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.446154,0.600000
2,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.697436,0.628931
3,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.569231,0.142857
4,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.671795,0.457627
5,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
6,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.584615,0.198020
7,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.743590,0.691358
8,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
9,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 4-Layers,0.569231,0.066667


In [ ]:
from tensorflow.keras.layers import Dropout
# Lade dein Daten
# Liste der kategorialen Spalten
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]



# One-Hot-Encoding für kategoriale Features
#categorical_features = ["Sex", "Ctry", "Town", "Goal of Training", "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
#X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# Feature Scaling
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Erstelle das neuronale Netz
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.5))  # 50% der Neuronen werden zufällig deaktiviert
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Dropout", "Neuronales Netz 4-Schichten 2Dropout 100 Epochen batchsize 50", y_pred_class, y_test)
df_results

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.4549 - loss: 192.3117 - val_accuracy: 0.5449 - val_loss: 83.9776
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4821 - loss: 121.3513 - val_accuracy: 0.5449 - val_loss: 26.5858
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4965 - loss: 85.0044 - val_accuracy: 0.5449 - val_loss: 9.3955
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5106 - loss: 73.1482 - val_accuracy: 0.4615 - val_loss: 26.0858
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5014 - loss: 58.9342 - val_accuracy: 0.4615 - val_loss: 16.1539
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5205 - loss: 48.8227 - val_accuracy: 0.4936 - val_loss: 10.3861
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5338 - loss: 42.9187 - val_accuracy: 0.4615 - val_loss: 14.6790
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5168 - loss: 34.8660 - val_accuracy: 0.4

,Durchlauf,Modell,Accuracy,F1 Score
0,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.507692,0.625000
1,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.446154,0.600000
2,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.697436,0.628931
3,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.569231,0.142857
4,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.671795,0.457627
5,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
6,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.584615,0.198020
7,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.743590,0.691358
8,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
9,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 4-Layers,0.569231,0.066667


In [ ]:
from tensorflow.keras.layers import Dropout
# Lade dein Daten
# Liste der kategorialen Spalten
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

# Wähle die Zielvariable und die Features
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]



# One-Hot-Encoding für kategoriale Features
#categorical_features = ["Sex", "Ctry", "Town", "Goal of Training", "Preferred Training Daytime", "Subscription Type", "Synchronisation"]
#X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# Feature Scaling
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Erstelle das neuronale Netz
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))  # 50% der Neuronen werden zufällig deaktiviert
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Dropout 0.2", "Neuronales Netz 4-Schichten 2Dropout 100 Epochen batchsize 50", y_pred_class, y_test)
df_results

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.4963 - loss: 103.3655 - val_accuracy: 0.6218 - val_loss: 3.4712
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5164 - loss: 51.7412 - val_accuracy: 0.5449 - val_loss: 46.2857
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5452 - loss: 36.2542 - val_accuracy: 0.5128 - val_loss: 13.7979
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5401 - loss: 29.3437 - val_accuracy: 0.4936 - val_loss: 13.0330
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5065 - loss: 22.9680 - val_accuracy: 0.4679 - val_loss: 5.4110
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5512 - loss: 16.7241 - val_accuracy: 0.5449 - val_loss: 3.0365
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5422 - loss: 14.0486 - val_accuracy: 0.5192 - val_loss: 4.9119
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5058 - loss: 9.9421 - val_accuracy: 0.5833 - 

,Durchlauf,Modell,Accuracy,F1 Score
0,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.507692,0.625000
1,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.446154,0.600000
2,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.697436,0.628931
3,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.569231,0.142857
4,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.671795,0.457627
5,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
6,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.584615,0.198020
7,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.743590,0.691358
8,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
9,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 4-Layers,0.569231,0.066667


In [ ]:
# Baseline-Modellierung
def create_model(seeds, layers, neurons, dropout_rate=0.0, batch_norm=False):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    for _ in range(layers):
        model.add(Dense(neurons, activation='relu',kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seeds)))
        if batch_norm:
            model.add(BatchNormalization())
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    return model

model = create_model(2856,4,32,0.2,True)
# Kompilieren des Modells
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Trainieren des Modells
model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=1)

# Evaluiere das Modell auf dem Testdatensatz
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {accuracy}")

# Vorhersagen auf dem Testdatensatz
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)  # Konvertiere die Vorhersagen in 0 oder 1

# Logge die Ergebnisse
df_results = log_model_performance("Dropout 0.2, seed 2856", "Neuronales Netz 4-Schichten 2Dropout 100 Epochen batchsize 50", y_pred_class, y_test)
df_results

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.5040 - loss: 0.8356 - val_accuracy: 0.4551 - val_loss: 1.1949
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6084 - loss: 0.7798 - val_accuracy: 0.4551 - val_loss: 0.9055
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6194 - loss: 0.7226 - val_accuracy: 0.4679 - val_loss: 0.8325
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6066 - loss: 0.7215 - val_accuracy: 0.4551 - val_loss: 0.9876
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6101 - loss: 0.6866 - val_accuracy: 0.5321 - val_loss: 0.9818
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6222 - loss: 0.7066 - val_accuracy: 0.5513 - val_loss: 0.7969
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6494 - loss: 0.7037 - val_accuracy: 0.6026 - val_loss: 0.7064
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6235 - loss: 0.6597 - val_accuracy: 0.5897 - 

,Durchlauf,Modell,Accuracy,F1 Score
0,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.507692,0.625000
1,Baseline,Neuronales Netz 2-Schichten 50 Epochen batchsize 10,0.446154,0.600000
2,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 2-Layers,0.697436,0.628931
3,"Baseline 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.569231,0.142857
4,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.671795,0.457627
5,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
6,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.584615,0.198020
7,"Best Seed Opt. 50-Epo, 10-BatchS",Neuronales Netz 3-Layers,0.743590,0.691358
8,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 3-Layers,0.738462,0.675159
9,"Best Seed Opt. 100-Epo, 50-BatchS",Neuronales Netz 4-Layers,0.569231,0.066667


In [ ]:
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
categorical_cols = ["Sex", "Ctry", "Town", "Goal of Training",
                    "Preferred Training Daytime", "Subscription Type", "Synchronisation"]

df = df_training_all_features.drop(columns=categorical_cols)  # Ersetze dies mit deiner Datei

#df = df.drop(columns=categorical_cols)
target_column = "User of latest model"
X = df.drop(columns=[target_column])
y = df[target_column]

# Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline-Modellierung
def create_model(seeds, layers, neurons, dropout_rate=0.0, batch_norm=False):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    for _ in range(layers):
        model.add(Dense(neurons, activation='relu',kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seeds)))
        if batch_norm:
            model.add(BatchNormalization())
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    return model


# Experimentiere mit verschiedenen Parametern
exp = []
seeds = np.random.randint(1, 100, size=10).tolist()
layers = [2, 3, 4]
neurons = [32, 64, 128]
dropout_rates = [0.0, 0.2, 0.5]
batch_norm_options = [True, False]

for seed in seeds:
    np.random.seed(seed)
    tf.random.set_seed(seed)
    for layer in layers:
        for neuron in neurons:
            for dropout_rate in dropout_rates:
                for batch_norm in batch_norm_options:
                    model = create_model(seeds, layer, neuron, dropout_rate, batch_norm)
                    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
                    model.fit(X_train, y_train, epochs=100, batch_size=50, validation_split=0.2, verbose=0)
                    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
                    exp.append({
                        "Seed": seed,
                        "Layers": layer,
                        "Neurons": neuron,
                        "Dropout Rate": dropout_rate,
                        "Batch Normalization": batch_norm,
                        "Test Accuracy": accuracy
                    })

# Ergebnisse in einem DataFrame ausgeben
exp_df = pd.DataFrame(exp)
exp_df

KeyboardInterrupt: 

In [ ]:
# Ergebnisse in einem DataFrame ausgeben
exp_df = pd.DataFrame(exp)
exp_df

,Seed,Layers,Neurons,Dropout Rate,Batch Normalization,Test Accuracy
0,2856,2,32,0.0,True,0.630769
1,2856,2,32,0.0,False,0.692308
2,2856,2,32,0.2,True,0.748718
3,2856,2,32,0.2,False,0.558974
4,2856,2,32,0.5,True,0.610256
...,...,...,...,...,...,...
56,2815,2,32,0.2,True,0.712821
57,2815,2,32,0.2,False,0.558974
58,2815,2,32,0.5,True,0.682051
59,2815,2,32,0.5,False,0.558974


from matplotlib import pyplot as plt
exp_df['Seed'].plot(kind='hist', bins=20, title='Seed')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
exp_df['Layers'].plot(kind='hist', bins=20, title='Layers')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
exp_df['Neurons'].plot(kind='hist', bins=20, title='Neurons')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
exp_df['Dropout Rate'].plot(kind='hist', bins=20, title='Dropout Rate')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(exp_df['Batch Normalization'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(exp_df, x='Layers', y='Batch Normalization', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(exp_df['Batch Normalization'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(exp_df, x='Neurons', y='Batch Normalization', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(exp_df['Batch Normalization'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(exp_df, x='Dropout Rate', y='Batch Normalization', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [15]:
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.callbacks import EarlyStopping
# Benutzerdefinierter Callback, um das Training zu stoppen, wenn die Accuracy in der ersten Epoche niedriger als 0.62 ist
class StopTrainingCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 and logs.get('accuracy') < 0.62:
            self.model.stop_training = True
            print("Stopping training due to low initial accuracy.")

early_stopping = EarlyStopping(
    monitor='val_loss',  # Überwache den Verlust auf der Validierungsmenge
    patience=10,         # Warte 10 Epochen, bevor das Training gestoppt wird
    restore_best_weights=True  # Wiederherstellung der besten Gewichte
)
